In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import seaborn as sb
import numpy as np
import h5py
from tqdm import tqdm
from scipy.optimize import curve_fit, minimize
from scipy.stats import t as tDistribution
from scipy.stats import multivariate_t
from scipy.special import gamma as gammaFunc
from scipy.integrate import quad

In [ ]:
datafolder = "data/"
imagefolder = "figures/"

In [ ]:
with h5py.File(datafolder+"UchuuLong.h5") as file:
	clusters = pd.DataFrame.from_records(file["clusters"][:], index="icl")
	galaxies = pd.DataFrame.from_records(file["galaxies"].fields(["x", "y", "z", "R", "r", "vz", "vR", "vTang", "upID", "icl"])[:])

In [ ]:
R_bin_width = 0.5
R_bins = np.arange(0, galaxies["R"].max() + R_bin_width/2, R_bin_width)
galaxies["Rbinned"] = pd.cut(galaxies["R"], bins=R_bins, labels=R_bins[:-1] + R_bin_width/2)

In [ ]:
N_M_bins = 5
mass_bins = np.geomspace(clusters["Mvcl"].min(), clusters["Mvcl"].max(), N_M_bins + 1)
clusters["Mbinned"] = pd.cut(clusters["Mvcl"], bins=mass_bins, labels=np.sqrt(mass_bins[:-1] * mass_bins[1:]))
galaxies["Mbinned"] = clusters.loc[galaxies["icl"], "Mbinned"].to_numpy()

In [ ]:
fig, ax = plt.subplots()
bins = np.geomspace(clusters["Mvcl"].min(), clusters["Mvcl"].max(), 30)
sb.histplot(clusters["Mvcl"], bins=bins, ax=ax)
for v in mass_bins:
	ax.axvline(v, color="r", ls="--")
ax.set_xlabel("Cluster Mass [M☉]")
ax.set_xscale("log")
ax.set_yscale("log")
fig.tight_layout()
fig.savefig(imagefolder+"cluster_mass_distribution.png")

In [ ]:
def normOfT(x, loc, scale, invdeg, skew, nevals=101, randomsize=100000):
	samples = multivariate_t.rvs(size=randomsize, loc=np.zeros(3), shape=scale**2*np.eye(3), df=1/invdeg)
	radial_distances = np.linalg.norm(samples[:,:2], axis=1)
	left_edge = x[0] - 0.5 * (x[1] - x[0])
	right_edge = x[-1] + 0.5 * (x[-1] - x[-2])
	counts, bins = np.histogram(radial_distances, bins=np.linspace(left_edge,right_edge,nevals), density=True)
	bin_centers = 0.5 * (bins[1:] + bins[:-1])
	return np.interp(x, bin_centers, counts, left=0, right=0)

def skewedTDistPDF(x, loc, scale, invdeg, skew):
	mu, sig, lam, q = loc, scale, skew, 1/(2*invdeg)
	if q < 100:
		ratio_of_gammas = gammaFunc(q - 0.5) / gammaFunc(q)
	else:
		ratio_of_gammas = q**(-0.5)
	v = 1 / np.sqrt(q * (1/(2*q-2)*(1+3*lam**2) - 4*lam**2/np.pi*ratio_of_gammas**2))
	m = lam * v * sig * 2 * np.sqrt(q/np.pi) * ratio_of_gammas
	if q < 100:
		ratio_of_gammas = gammaFunc(0.5 + q) / gammaFunc(q)
	else:
		ratio_of_gammas = q**0.5
	val = ratio_of_gammas / v / sig / np.sqrt(np.pi*q) / (1 + (x-mu+m)**2 / (q*v**2*sig**2*(1+lam*np.where(x-mu+m< 0,-1,1))**2))**(0.5+q)
	if not np.isfinite(val).all():
		print(f"Fitting parameters: loc={loc}, scale={scale}, invdeg={invdeg}, skew={skew}", flush=True, end='; ')
		print(f"q={q}", flush=True)
	return val

def residuals(params, counts_vR, bin_centers_vR, counts_vTang, bin_centers_vTang):
	return np.sum((counts_vR - skewedTDistPDF(bin_centers_vR, *params))**2) + np.sum((counts_vTang - normOfT(bin_centers_vTang, *params))**2)

def fitParametersOnlyRadial(data):
	counts, bins = np.histogram(data, bins=np.linspace(-4, 4, 101), density=True)
	bin_centers = 0.5 * (bins[1:] + bins[:-1])
	initial_guess = [0.0, 1.0, 0.2, 0.0]
	params, _ = curve_fit(skewedTDistPDF, bin_centers, counts, p0=initial_guess, bounds=([-4, 0.01, 5e-3, -0.99], [4, 5.0, 0.5, 0.99]))
	return params

def fitParametersJoint(data):
	counts_vR, bins_vR = np.histogram(data["vR"], bins=np.linspace(-4, 4, 101), density=True)
	bin_centers_vR = 0.5 * (bins_vR[1:] + bins_vR[:-1])
	counts_vTang, bins_vTang = np.histogram(data["vTang"], bins=np.linspace(0, 4, 101), density=True)
	bin_centers_vTang = 0.5 * (bins_vTang[1:] + bins_vTang[:-1])
	x0 = [0.0, 1.0, 0.2, 0.0]
	result = minimize(residuals, x0, args=(counts_vR, bin_centers_vR, counts_vTang, bin_centers_vTang), method="Powell", bounds=[(-4,4), (0.01,5.0), (5e-3,0.499), (-0.99,0.99)])
	params = result.x
	return params

def fitFamily(df):
	results = []
	for (Mbin,Rbin), group in tqdm(df.groupby(["Mbinned","Rbinned"], observed=False)):
		params = fitParametersJoint(group[["vR", "vTang"]])
		results.append([Mbin,Rbin] + params.tolist())
	return pd.DataFrame(results, columns=["Mbinned", "Rbinned", "loc", "scale", "invdeg", "skew"]).set_index(["Mbinned", "Rbinned"])

In [ ]:
fit_params = fitFamily(galaxies)
fit_params

In [ ]:
fit_params.to_csv("fit_params.csv", index=True)

In [ ]:
fit_params = pd.read_csv("fit_params.csv", index_col=["Mbinned", "Rbinned"])
fit_params

In [ ]:
avged = fit_params.groupby("Rbinned", observed=False).mean()
avged

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	ax = axs[i]
	ax.hist(group["vR"], bins=np.linspace(-3,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(-3, 3, 1001)
	ax.plot(xs, skewedTDistPDF(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vR [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vR_distribution_0-5.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	if Rbin < 5:
		continue
	ax = axs[i]
	ax.hist(group["vR"], bins=np.linspace(-3,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(-3, 3, 1001)
	ax.plot(xs, skewedTDistPDF(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vR [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vR_distribution_5-10.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	if Rbin < 10:
		continue
	ax = axs[i]
	ax.hist(group["vR"], bins=np.linspace(-3,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(-3, 3, 1001)
	ax.plot(xs, skewedTDistPDF(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vR [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vR_distribution_10-15.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	if Rbin < 15:
		continue
	ax = axs[i]
	ax.hist(group["vR"], bins=np.linspace(-3,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(-3, 3, 1001)
	ax.plot(xs, skewedTDistPDF(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vR [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vR_distribution_15-20.png")

In [ ]:
from hubbleflow.fittingfunctions import velocityCorrection

In [ ]:
init_guess = [-6., -2., 3., -0.5, 5., 1.5, 1/10]
fitting_curve = curve_fit(velocityCorrection, avged.index, avged["loc"].to_numpy(), p0=init_guess, bounds=([-1e3,-1e2,1e-3,-1e1,1e-1,0,0], [-1e-3,-1e-2,1e1,-1e-5,1e2,5,1/10]), maxfev=10000)[0]
print(f"a={fitting_curve[0]:.2f}, b={fitting_curve[1]:.2f}, c={fitting_curve[2]:.2f}, m={fitting_curve[3]:.2f}, s={fitting_curve[4]:.2f}, d={fitting_curve[5]:.2f}, h={fitting_curve[6]:.2f}")

In [ ]:
xs = np.linspace(0, 20, 2000)
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
color_normalizer = mpl.colors.LogNorm(vmin=fit_params.reset_index()["Mbinned"].min(), vmax=fit_params.reset_index()["Mbinned"].max())
scplt = sb.scatterplot(data=fit_params.reset_index(), x="Rbinned", y="loc", hue="Mbinned", palette="crest", hue_norm=color_normalizer, s=100, alpha=0.7, ax=axs[0])
Rbinned = avged.index.get_level_values("Rbinned")
axs[0].plot(Rbinned, avged["loc"], color="r", marker="*", label="Average fitted values")
axs[1].plot(Rbinned, avged["loc"], color="r", marker="*", label="Average fitted values")
axs[1].plot(xs, velocityCorrection(xs, *fitting_curve), label="Best fit")
axs[1].plot(xs, 1/10*xs-0.75, label="Shifted Hubble Flow")
max_mass = fit_params.index.get_level_values("Mbinned").max()
axs[1].errorbar(Rbinned, avged["loc"], yerr=fit_params["scale"].loc[max_mass], color="r", lw=0.3)
for ax in axs:
    ax.legend(loc="upper left")
    ax.set_xlabel(r"$R$ [Virial Units]")
axs[0].set_ylabel(r"$v_R$ [Virial Units]")
axs[0].set_ylim([-0.75,1.5])
axs[0].set_title("Location parameter fitted values")
axs[1].set_title("Location parameter models")
fig.tight_layout()
fig.savefig(imagefolder+"vR_correction.png")

In [ ]:
from hubbleflow.fittingfunctions import logSigmaFittingCurve

data_fit = fit_params.loc[(slice(None), slice(5, None)), :].copy()
data_fit = data_fit.groupby(level="Mbinned", observed=False).mean()
sigma_fit = curve_fit(logSigmaFittingCurve, data_fit.index.to_numpy(), np.log(data_fit["scale"]).to_numpy(), p0=[14, np.log(0.46)], bounds=([10, -np.inf], [20, np.inf]))[0]
sigma_fit

In [ ]:
fig, ax = plt.subplots()
data = fit_params.loc[(slice(None), slice(5, None)), :].copy()
data = data.groupby(level="Mbinned", observed=False).mean()
ax.scatter(data.index, np.log(data["scale"]), s=50, color="r", label="Fitted scale values")
xs = np.linspace(clusters["Mvcl"].min(), clusters["Mvcl"].max(), 200)
ax.plot(xs, logSigmaFittingCurve(xs, *sigma_fit), color="b", label="Model fit")
ax.legend(loc="lower left")
ax.set_xscale("log")
ax.set_xlabel("Cluster Virial Mass [M☉]")
ax.set_ylabel("Log Scale Parameter")
ax.set_title("Scale Parameter dependency on Cluster Mass")
fig.tight_layout()
fig.savefig(imagefolder+"fit_scale_parameter_dependency.png")


In [ ]:
from hubbleflow.fittingfunctions import scaleFittingCurve
def scaleFittingCurveScipy(MandR, f_0, a):
	return scaleFittingCurve(MandR[:,1], MandR[:,0], f_0, a, *sigma_fit)

In [ ]:
scale_fit = curve_fit(scaleFittingCurveScipy, fit_params.reset_index()[["Mbinned", "Rbinned"]].to_numpy(), fit_params["scale"].to_numpy(), p0=[0.7, 0.5], bounds=([0, 0], [2, 10]))[0]
scale_fit

In [ ]:
from hubbleflow.fittingfunctions import invdegFittingCurve

In [ ]:
invdeg_fit = curve_fit(invdegFittingCurve, fit_params.index.get_level_values("Rbinned").to_numpy(), fit_params["invdeg"].to_numpy(), p0=[0.5, 0.5, 2., 1., 5., 1.5], bounds=([0, 0, 1e-3, 1e-3, 0, 0], [5, 5, 1e2, 1e2, 1e2, 5]))[0]
invdeg_fit

In [ ]:
from hubbleflow.fittingfunctions import skewFittingCurve

In [ ]:
skew_fit = curve_fit(skewFittingCurve, fit_params.index.get_level_values("Rbinned").to_numpy(), fit_params["skew"].to_numpy(), p0=[-0.1, 1., 0.1], bounds=([-10, 0, 0], [0, 5, 10]))[0]
skew_fit

In [ ]:
import json
fitting_results = {
	"location": {
		"parameters": fitting_curve.tolist(),
		"function": "velocityCorrection(R, a, b, c, m, s, d, h)"
	},
	"scale": {
		"parameters": scale_fit.tolist() + sigma_fit.tolist(),
		"function": "scaleFittingCurve(M, R, f_0, a, f_c, f_0)"
	},
	"inverse_degrees_of_freedom": {
		"parameters": invdeg_fit.tolist(),
		"function": "invdegFittingCurve(R, p0, p1, p2, p3, p4, p5)"
	},
	"skewness": {
		"parameters": skew_fit.tolist(),
		"function": "skewFittingCurve(R, k0, k1, k2)"
	}
}
with open("fitting_results.json", "w") as f:
	json.dump(fitting_results, f, indent=4)

In [ ]:
#fig, axs = plt.subplots(2, 2, figsize=(10, 6))
fig = plt.figure(figsize=(15, 6))
gs = mpl.gridspec.GridSpec(2, 3, figure=fig, width_ratios=[1,1,0.03])
axs = [fig.add_subplot(gs[i,j]) for i in range(2) for j in range(2)]
cax = fig.add_subplot(gs[:,2])
color_normalizer = mpl.colors.LogNorm(vmin=fit_params.reset_index()["Mbinned"].min(), vmax=fit_params.reset_index()["Mbinned"].max())
for i, col in enumerate(fit_params.columns):
	ax = axs[i]
	data = fit_params.reset_index()
	scplt = sb.scatterplot(data=data, x="Rbinned", y=col, hue="Mbinned", palette="viridis", hue_norm=color_normalizer, legend=False, alpha=0.7, ax=ax)
	ax.plot(avged.index, avged[col], color="k", lw=2, label="Average over Mass Bins")
	xs = np.linspace(avged.index.get_level_values("Rbinned").min(), avged.index.get_level_values("Rbinned").max(), 100)
	if col == "loc":
		ax.plot(xs, velocityCorrection(xs, *fitting_curve), color="r", label="Fitting Curve to Averages")
	elif col == "scale":
		for Mbin in fit_params.index.get_level_values("Mbinned").unique():
			ax.plot(xs, scaleFittingCurve(xs, Mbin, *scale_fit, *sigma_fit), color="r")
		ax.plot([], [], color="r", label="Fitting Curve to Mass-dependent curves")
	elif col == "invdeg":
		ax.plot(xs, invdegFittingCurve(xs, *invdeg_fit), color="r", label="Fitting Curve to Averages")
	elif col == "skew":
		ax.plot(xs, skewFittingCurve(xs, *skew_fit), color="r", label="Fitting Curve to Averages")
	ax.legend()
	ax.set_xlabel("R bin center [Virial Units]")
	ax.set_ylabel(col)
colormap = plt.cm.ScalarMappable(norm=color_normalizer, cmap="viridis")
cbar = fig.colorbar(colormap, cax=cax, ticks=np.unique(fit_params.index.get_level_values("Mbinned")))
fig.tight_layout()
fig.savefig(imagefolder+"fitted_parameters_overview.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	ax = axs[i]
	ax.hist(group["vTang"], bins=np.linspace(0,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(0, 3, 1001)
	ax.plot(xs, normOfT(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vTang [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vTang_distribution_0-5.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	if Rbin < 5:
		continue
	ax = axs[i]
	ax.hist(group["vTang"], bins=np.linspace(0,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(0, 3, 1001)
	ax.plot(xs, normOfT(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vTang [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vTang_distribution_5-10.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	if Rbin < 10:
		continue
	ax = axs[i]
	ax.hist(group["vTang"], bins=np.linspace(0,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(0, 3, 1001)
	ax.plot(xs, normOfT(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vTang [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vTang_distribution_10-15.png")

In [ ]:
fig, axs = plt.subplots(10, N_M_bins, figsize=(4*N_M_bins, 2.5*10), sharex=True)
axs = axs.flatten()
i = 0
for (Rbin,Mbin), group in galaxies.groupby(["Rbinned","Mbinned"], observed=False):
	if Rbin < 15:
		continue
	ax = axs[i]
	ax.hist(group["vTang"], bins=np.linspace(0,3,101), density=True, alpha=0.5, label="Data")
	xs = np.linspace(0, 3, 1001)
	ax.plot(xs, normOfT(xs, *fit_params.loc[(Mbin, Rbin)]), color="r", label="Fit")
	ax.set_title(f"Mbin={Mbin:.2e}, Rbin={Rbin:.2f}")
	if i % N_M_bins == 0:
		ax.set_ylabel("Density")
	if i >= (R_bins.shape[0]-2)*N_M_bins:
		ax.set_xlabel("vTang [Virial Units]")
	i += 1
	if i == len(axs):
		break
fig.tight_layout()
fig.savefig(imagefolder+"vTang_distribution_15-20.png")

In [ ]:
from hubbleflow.kerneldensityestimation import KernelDensityEstimation

In [ ]:
from hubbleflow.fittingfunctions import PVcorrclass, PUclass, unnormalizedPosteriorClass

In [ ]:
H0 = 1 / 10

PVcorr = PVcorrclass(*fitting_curve)

PU_args = np.concatenate((scale_fit, sigma_fit, invdeg_fit))
PU = PUclass(*PU_args)

Rhist, Rbins = np.histogram(galaxies["R"], bins=100, density=True)

binwidth = 0.01
bins = np.arange(-binwidth/2, 20+binwidth, binwidth)
PR = KernelDensityEstimation(galaxies["R"].to_numpy(), bins, bandwidth=binwidth/2*1.4, a=0, b=20)

unnormalizedPosterior = unnormalizedPosteriorClass(PR, PVcorr, PU, H0)

def normalizedPosterior(R, r, v, M):
	if np.isscalar(R):
		if R < r:
			return 0
		unnormalized_value = unnormalizedPosterior(R, r, v, M)
		normalization = quad(unnormalizedPosterior, r, 20, args=(r, v, M))[0]
		if normalization == 0:
			return 0
		return unnormalized_value / normalization
	value = unnormalizedPosterior(R, r, v, M)
	normalization = quad(unnormalizedPosterior, r, 20, args=(r, v, M))[0]
	if normalization == 0:
		return 0
	return value / normalization
def firstComponent(R, r, v):
	value = np.zeros_like(R)
	mask = R >= r
	value[mask] = r/R[mask]/np.sqrt(R[mask]**2-r**2)
	return value
def secondComponent(R, r, v, M):
	value = np.zeros_like(R)
	mask = R >= r
	value[mask] = PU(np.abs(v)-(PVcorr(R[mask]))*(np.cos(np.arcsin(r/R[mask]))), R[mask], M)
	return value

In [ ]:
n_rbins, n_vzbins = 10, 10
rbins = np.linspace(galaxies["R"].min(), galaxies["R"].max(), n_rbins+1)
vzbins = np.linspace(galaxies["vz"].min(), galaxies["vz"].max(), n_vzbins+1)

In [ ]:
bounds = [
	[[1.5,1.6],[-0.0125,0.0125]],
	[[2,3],[-0.0125,0.0125]],
	[[5,6],[-0.01,0.01]],
	[[6,7],[1,1.02]],
]

fig, axs = plt.subplots(len(bounds), 1, figsize=(6, 3*len(bounds)))

idx = 0
for idx, ((rbins_min, rbins_max), (vzbins_min, vzbins_max)) in tqdm(enumerate(bounds)):
		ax = axs[idx]
		rbin_center = 0.5 * (rbins_min + rbins_max)
		vzbin_center = 0.5 * (vzbins_min + vzbins_max)
		sliced = galaxies.loc[(galaxies["r"] >= rbins_min) & (galaxies["r"] < rbins_max) & (galaxies["vz"] >= vzbins_min) & (galaxies["vz"] < vzbins_max)]
		Ms = clusters.loc[sliced["icl"], "Mvcl"].to_numpy()
		rs, vzs = sliced[["r","vz"]].to_numpy().T
		if Ms.shape[0] == 0:
			continue
		Rs = np.linspace(0, 20, 101)
		delta_R = Rs[1] - Rs[0]
		posterior = np.array([normalizedPosterior(Rs, r, vz, M) for r,vz,M in zip(rs, vzs, Ms)]).mean(axis=0)
		ax.plot(Rs, posterior, color="b", label="Posterior")
		ax.hist(sliced["R"], bins=Rs, density=True, alpha=0.5, label="Data", color="gray")
		ax.set_title(f"r={rbin_center:.2f}, vz={vzbin_center:.2f}")
		ax.set_xlabel("R [Virial Units]")
		ax.set_ylabel("Density")
		ax.legend(loc="upper right")
		idx += 1
fig.tight_layout()
fig.savefig(imagefolder+"posterior_R_distributions.png")

In [ ]:
sampled = galaxies.sample(5)
sampled

In [ ]:
Rs = np.linspace(0, 30, 1001)
deltaR = Rs[1] - Rs[0]
fig, axs = plt.subplots(5, 1, figsize=(6, 3*5))
for i in range(5):
	ys = unnormalizedPosterior(Rs, sampled["r"].iloc[i], sampled["vz"].iloc[i], clusters.loc[sampled["icl"].iloc[i], "Mvcl"])
	ys = ys / (ys*deltaR).sum()
	axs[i].plot(Rs, ys, label="Unnormalized Posterior")
	axs[i].axvline(sampled["R"].iloc[i], label="Observed R", color="r", ls="--")
	ys = unnormalizedPosterior.geometricComponent(Rs, sampled["r"].iloc[i])
	ys = ys / (ys*deltaR).sum()
#	axs[i].plot(Rs, ys, label="Geometric Component")
	ys = unnormalizedPosterior.velocityComponent(Rs, sampled["r"].iloc[i], sampled["vz"].iloc[i], clusters.loc[sampled["icl"].iloc[i], "Mvcl"])
	ys = ys / (ys*deltaR).sum()
#	axs[i].plot(Rs, ys, label="Velocity Component")
	axs[i].legend()